In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window as w

spark = SparkSession.builder.appName('task_1').getOrCreate()

employees = spark.read.csv('/Volumes/mycatalog/myschema/task_1/emp_data_.csv', header = True, inferSchema = True)

orders = spark.read.csv('/Volumes/mycatalog/myschema/task_1/orders.csv', header = True, inferSchema = True)

# select loc as location from orders

orders = orders.select(col('order_id'), 
                       col('product_name'), 
                       col('quantity'),
                       col('loc').alias('location'),
                       col('product_price'),
                       col('sell_price'),
                       col('availablity_status'),
                       col('mob_no'),
                       col('id').alias('customer_id'))

employees = employees.select(col('empid'), 
                             col('fname'), 
                             col('lname'),
                             col('gender'),
                             col('salary'),
                             col('old_salary'),
                             col('dept_name'),
                             col('dept_id'),
                             col('loc').alias('location'))

employees.printSchema()

In [0]:
employee_schema = StructType([
    StructField('Employee_ID', IntegerType()),
    StructField('First_name', StringType()),
    StructField('Last_name', StringType()),
    StructField('Gender', StringType()),
    StructField('Current_Salary', IntegerType()),
    StructField('Old_slary', IntegerType()),
    StructField('Department_name', StringType()),
    StructField('Department_ID', IntegerType()),
    StructField('Location', StringType())
])

orders_schema = StructType([
    StructField('Order_ID', IntegerType()),
    StructField('Product', StringType()),
    StructField('Quantity', IntegerType()),
    StructField('Location', StringType()),
    StructField('Product_price', IntegerType()),
    StructField('Selling_price', IntegerType()),
    StructField('Availability_status', StringType()),
    StructField('Mobile_No', LongType()),
    StructField('Employee_ID', IntegerType())
])

orders = spark.read.schema(orders_schema).csv('/Volumes/mycatalog/myschema/task_1/orders.csv', header = True)

employees = spark.read.schema(employee_schema).csv('/Volumes/mycatalog/myschema/task_1/emp_data_.csv', header = True)

def get_rds_to_s3_transformed_data(orders ,employees):
    
    # employees.join(orders, orders.Employee_ID == employees.Employee_id)
    output = employees.join(orders, on = 'Employee_ID')\
                      .groupBy(orders.Location)\
                      .agg(count('*').alias('total'))

    return output

transformed_data = get_rds_to_s3_transformed_data(orders, employees)

transformed_data.write.mode('overwrite').csv('s3://pyspark-csv-1-1/mycsv/task1', header = True)

# display(transformed_data)